DERTERMINING THE OPTIMAL NO. OF HIDDEN LAYERS AND NEURONS IN AN ANN 

##METHODS TO FOLLOW 

START SIMPLE: begin with simple architecture and increase complexity

GRID SEARCH/RANDOM SEARCH : use grod or random serach to try different architectures

CROSS VALIDATION : use cross validation to evaluate the performance of dufferent architecture 

HEURISTICS AND RULES OF THUMB : some heuristics and empirical rules can provide starting points such as 

(1)  the number of neurons in the hidden layer should be between the size of the input layer and the size of the output layer 

(2) A common practice is to start with 1 2 hidden layers

In [3]:
import pandas as pd 
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, LabelEncoder,OneHotEncoder
from sklearn.pipeline import Pipeline
from scikeras.wrappers import KerasClassifier
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.compose import ColumnTransformer
import pickle

In [4]:
data=pd.read_csv('Churn_MOdelling.csv')

In [5]:
label_encoder=LabelEncoder()
data['Gender']=label_encoder.fit_transform(data['Gender'])
preprocessor=ColumnTransformer(
    transformers=[
        ('onehot',OneHotEncoder(drop='first'),['Geography']),
        ('scaler',StandardScaler(),['CreditScore','Age','Tenure','Balance','NumOfProducts'])
    ],
    remainder='passthrough'
)

In [6]:

X=data.drop(columns=['EstimatedSalary'])
y=data['EstimatedSalary']

In [7]:
X_train,X_test,y_train,y_test=train_test_split(X,y,random_state=42,test_size=0.2)

In [8]:
X_train_scaled=preprocessor.fit_transform(X_train)

In [9]:
X_test_scaled=preprocessor.transform(X_test)

In [10]:
with open('labelencoderregression.pkl','wb') as file:
    pickle.dump(label_encoder,file)


with open('preprocessor_regression.pkl','wb') as file:
    pickle.dump(preprocessor,file)    

In [11]:
##Define a function to try different parameters(kerasclassifier)
def create_model(neurons=32,layers=1):
    model=Sequential()
    model.add(Dense(neurons,activations='relu',input_shape=(X_train_scaled.shape[1],)))

    for _ in range(layers-1):
        model.add(Dense(neurons,activation='relu'))

    model.add(Dense(1,activation='sigmoid'))
    model.compile(optimizer='adam',loss="binary_crossentropy",metrics=['accuracy'])

    return model

In [12]:
# create a keras classifier 
model=KerasClassifier(build_fn=create_model,epochs=50,batch_size=10,verbose=0)

In [13]:
#de3fine parameters
param_grid={
    'neurons':[16,32,64,128],
    'layers':[1,2],
    'epochs':[50,100]
}

In [15]:
from sklearn.model_selection import GridSearchCV

grid = GridSearchCV(
    estimator=model,
    param_grid=param_grid,
    n_jobs=-1,
    cv=3
)

grid_result = grid.fit(X_train_scaled, y_train)

print("Best Score: %f using %s" % (
    grid_result.best_score_,
    grid_result.best_params_
))

ValueError: Invalid parameter layers for estimator KerasClassifier.
This issue can likely be resolved by setting this parameter in the KerasClassifier constructor:
`KerasClassifier(layers=1)`
Check the list of available parameters with `estimator.get_params().keys()`